# 🧠 Classificador de Conversas — Sentimento + Tags (Multi-Label)

**Projeto de portfólio para o trilho GCP · AI Pipelines**

Este notebook é um *esqueleto* para explorar a ideia no Google Colab:
- Classificação de **sentimento** (single-label): `positivo` / `neutro` / `negativo`
- Classificação de **tags** (multi-label): `reembolso`, `login`, `problema_técnico`, etc.

Aqui vamos fazer o **Milestone M1**: exploração + baseline (TF-IDF + Logistic Regression).

> ⚠️ Dataset: ["Customer Support on Twitter" (twcs)](https://www.kaggle.com/thoughtvector/customer-support-on-twitter) — download externo.


## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn


## 2. Carregar dados (exemplo com CSV local)

In [ ]:
import pandas as pd
import numpy as np

# Substitua pelo caminho do seu CSV baixado do twcs
df = pd.read_csv("data/raw/twcs.csv")
df.head()


## 3. Exploração

In [ ]:
df['sentiment'].value_counts()
print("\n---")
df['topic'].value_counts().head(15)


## 4. Pré-processamento

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

# Texto limpo (exemplo simples)
df['text_clean'] = df['text'].str.lower().str.replace('[^a-zà-ÿ\\s]', '', regex=True)

# Split
X = df['text_clean']
y_sentiment = df['sentiment']
y_tags = df['topic'].apply(lambda x: x.split('|') if isinstance(x, str) else [])

X_train, X_test, y_s_train, y_s_test, y_t_train, y_t_test = train_test_split(
    X, y_sentiment, y_tags, test_size=0.2, random_state=42, stratify=y_sentiment
)

# Vectorizer
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

# Multi-label binarizer
mlb = MultiLabelBinarizer()
y_t_train_enc = mlb.fit_transform(y_t_train)
y_t_test_enc = mlb.transform(y_t_test)

print(f"Shape treino: {X_train_vec.shape}")
print(f"Tags ({len(mlb.classes_)}): {list(mlb.classes_)}")


## 5. Modelo — Sentimento (baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

clf_s = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_s.fit(X_train_vec, y_s_train)
y_s_pred = clf_s.predict(X_test_vec)

print("=== SENTIMENTO ===")
print(f"Accuracy: {accuracy_score(y_s_test, y_s_pred):.4f}")
print(f"F1 macro: {f1_score(y_s_test, y_s_pred, average='macro'):.4f}")
print(classification_report(y_s_test, y_s_pred))


## 6. Modelo — Tags (multi-label)

In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss

clf_t = MultiOutputClassifier(LogisticRegression(max_iter=1000))
clf_t.fit(X_train_vec, y_t_train_enc)
y_t_pred = clf_t.predict(X_test_vec)

print("=== TAGS (MULTI-LABEL) ===")
print(f"Hamming loss: {hamming_loss(y_t_test_enc, y_t_pred):.4f}")

# Sample-level F1
from sklearn.metrics import f1_score
print(f"F1 macro (samples): {f1_score(y_t_test_enc, y_t_pred, average='samples'):.4f}")


## 7. Predição manual

In [ ]:
def predict(text):
    v = vec.transform([text.lower()])
    sent = clf_s.predict(v)[0]
    tags_raw = clf_t.predict(v)[0]
    tags = [mlb.classes_[i] for i, val in enumerate(tags_raw) if val]
    return {"sentiment": sent, "tags": tags}

print(predict("Não consigo fazer login, o sistema dá erro!"))
print(predict("Obrigado pelo atendimento, tudo perfeito!"))


## 8. Próximos passos (M2–M7)

In [ ]:
# M2: trocar por DistilBERT (Hugging Face) para ganho de accuracy
# M3: FastAPI + Docker
# M4: Cloud Run
# M5: Pipeline agendado (Cloud Scheduler → BigQuery)
# M6: Dashboard (Looker Studio / Streamlit)
# M7: README + deploy
